## Simple Chat Bot

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print ("Impoted Libraries")

Impoted Libraries


In [3]:
raw = pd.read_excel("GFC_10K_Financial_Analysis.xlsx", sheet_name="Raw Data")
ratios = pd.read_excel("GFC_10K_Financial_Analysis.xlsx", sheet_name="Ratios")
raw.head()

,Company,FiscalYear,Reporting Date,Revenue ($mm),GrossProfit ($mm),R&D Expense ($mm),SG&A Expense ($mm),Total OpEx ($mm),Operating Income ($mm),Net Income ($mm),...,Cash & Equivalents ($mm),Total Current Assets ($mm),Total Assets ($mm),Total Current Liabilities ($mm),Total Liabilities ($mm),Total Equity ($mm),Operating Cash Flow ($mm),CapEx ($mm),Prior Yr Revenue ($mm),Source / Notes
0,Apple,FY2023,2023-09-30,383285,169148,29915,24932,54847,114301,96995,...,29965,143566,352583,145308,290437,62146,110543,10959,394328,"Apple FY2023 10-K (SEC EDGAR, filed 2023-11-03..."
1,Apple,FY2024,2024-09-28,391035,180683,31370,26097,57467,123216,93736,...,29943,152987,364980,176392,308030,56950,118254,9447,383285,"TODO: fill from 10-K (SEC EDGAR, filed 2024-08..."
2,Apple,FY2025,2025-09-27,416161,195201,34550,27601,62151,133050,112010,...,35934,147957,359241,165631,285508,73733,111482,12715,391035,"TODO: fill from 10-K (SEC EDGAR, filed 2025-10..."
3,Microsoft,FY2023,2023-06-30,211915,146052,27195,30334,57529,88523,72361,...,34704,184257,411976,104149,205753,206223,87582,28107,198270,"TODO: fill from 10-K (SEC EDGAR, filed 2023-07..."
4,Microsoft,FY2024,2024-06-30,245122,171008,29510,31525,61035,109433,88136,...,18315,159734,512163,125286,243686,268477,118548,44477,211915,"TODO: fill from 10-K (SEC EDGAR, filed 2027-07..."


## Predefined queries

In [ ]:
# Query 1: What is each company's total revenue?(the canned response should list the revenue across all 3 years for each company)
def get_total_revenue():
    lines = []
    for company in raw['Company'].unique():
        company_data = raw[raw['Company'] == company].sort_values('FiscalYear ')
        lines.append(f"{company}:")
        for _, row in company_data.iterrows():
            lines.append(f"  FY{int(row['FiscalYear '])}: ${row['Revenue ($mm)']:,.0f}M")
    return "\n".join(lines)


In [ ]:
# Query 2: What is the overall financial trend for each company between FY2023 and FY2025?
def get_trend_summary():
    verdicts = {
        "Apple": "Stable-Improving",
        "Microsoft": "Strong",
        "Tesla": "Weakening"
    }
    lines = []
    for company in ratios['Company'].unique():
        company_data = ratios[ratios['Company'] == company].sort_values('FiscalYear')
        fy2023 = company_data[company_data['FiscalYear'] == 2023].iloc[0]
        fy2025 = company_data[company_data['FiscalYear'] == 2025].iloc[0]

        lines.append(f"{company}: {verdicts[company]}")
        lines.append(f"  Net Margin: {fy2023['Net Margin %']:.1f}% (FY2023) -> {fy2025['Net Margin %']:.1f}% (FY2025)")
        lines.append(f"  Revenue Growth: {fy2023['Revenue Growth % (YoY)']:.1f}% (FY2023) -> {fy2025['Revenue Growth % (YoY)']:.1f}% (FY2025)")
    return "\n".join(lines)

In [ ]:
# Query 3: Which company had the highest revenue growth?
def get_highest_growth():
    fy2025 = ratios[ratios['FiscalYear'] == 2025]
    top = fy2025.loc[fy2025['Revenue Growth % (YoY)'].idxmax()]
    return (f"{top['Company']} had the highest revenue growth in FY2025, "
            f"at {top['Revenue Growth % (YoY)']:.1f}% year-over-year.")


In [ ]:
# Query 4: Which company's growth trajectory changed the most?
def get_biggest_trajectory_change():
    results = []
    for company in ratios['Company'].unique():
        company_data = ratios[ratios['Company'] == company]
        fy2023_growth = company_data[company_data['FiscalYear'] == 2023]['Revenue Growth % (YoY)'].iloc[0]
        fy2025_growth = company_data[company_data['FiscalYear'] == 2025]['Revenue Growth % (YoY)'].iloc[0]
        change = fy2025_growth - fy2023_growth
        results.append((company, fy2023_growth, fy2025_growth, change))

    biggest = max(results, key=lambda x: abs(x[3]))
    company, fy23, fy25, change = biggest
    direction = "decline" if change < 0 else "acceleration"
    return (f"{company}'s growth trajectory changed the most, "
            f"from {fy23:.1f}% (FY2023) to {fy25:.1f}% (FY2025) — "
            f"a {abs(change):.1f} percentage point {direction}.")

In [ ]:
# Query 5: How does profitability compare across the three companies?
def get_profitability_comparison():
    fy2025 = ratios[ratios['FiscalYear'] == 2025].sort_values('Net Margin %', ascending=False)
    lines = ["Net Margin comparison, FY2025:"]
    for _, row in fy2025.iterrows():
        lines.append(f"  {row['Company']}: {row['Net Margin %']:.1f}%")
    return "\n".join(lines)

In [ ]:
def simple_chatbot(user_query):
    query = user_query.lower().strip()
    if "total revenue" in query:
        return get_total_revenue()
    elif "trend" in query:
        return get_trend_summary()
    elif "highest revenue growth" in query or "fastest growing" in query:
        return get_highest_growth()
    elif "trajectory" in query or "reversing" in query or "changed the most" in query:
        return get_biggest_trajectory_change()
    elif "profitability" in query or "profit margin" in query or "margins compare" in query:
        return get_profitability_comparison()
    else:
        return "Sorry, I can only provide information on predefined queries."

In [ ]:
simple_chatbot ("What is each company's total revenue?")